In [ ]:
import os
import gdown
import shutil
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import cv2
import random
import tensorflow as tf

from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Dropout,
    Conv2DTranspose, concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from sklearn.model_selection import train_test_split

In [ ]:
source = "/content/drive/MyDrive/satalite data/data"

destination = "/content/SatelliteDataset"

shutil.copytree(source, destination)



In [ ]:
with rasterio.open("/content/SatelliteDataset/images/0.tif") as src:
    print("Bands:", src.count)
    print("Shape:", src.height, src.width)
    print("Data type:", src.dtypes)

In [ ]:


def preview_satellite_image_grid(image_path):

    with rasterio.open(image_path) as src:
        image = src.read()  # shape: (12, H, W)

    band_names = [
        "Coastal", "Blue", "Green", "Red",
        "NIR", "SWIR1", "SWIR2",
        "QA Band", "Merit DEM", "Copernicus DEM",
        "ESA World Cover", "Water Occurrence Probability"
    ]

    # Prepare figure
    fig, axes = plt.subplots(2, 5, figsize=(25, 10))
    axes = axes.flatten()

    # 1️⃣ RGB composite in first subplot
    rgb = image[[3,2,1]].astype(np.float32)
    for c in range(3):
        band = rgb[c]
        rgb[c] = (band - band.min()) / (band.max() - band.min())
    rgb = np.transpose(rgb, (1,2,0))

    axes[0].imshow(rgb)
    axes[0].set_title("RGB Composite")
    axes[0].axis("off")

    # Indices to show: 0, 4, 5, 6, 7, 8, 9, 10, 11
    channel_indices = [0, 4, 5, 6, 7, 8, 9, 10, 11]

    for ax_idx, i in enumerate(channel_indices, start=1):
        band = image[i].astype(np.float32)
        name = band_names[i]

        # Choose colormap and normalization
        if i <= 6:  # spectral bands
            band = (band - band.min()) / (band.max() - band.min())
            cmap = "Blues"
        elif i in [8,9]:  # DEMs
            band = (band - band.min()) / (band.max() - band.min())
            cmap = "Blues"
        elif i == 11:  # water probability
            band = band / 100.0
            cmap = "Blues"
        else:  # QA / World Cover
            cmap = "Blues"

        axes[ax_idx].imshow(band, cmap=cmap)
        axes[ax_idx].set_title(name)
        axes[ax_idx].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
folder = "/content/SatelliteDataset/images"
files = sorted([f for f in os.listdir(folder) if f.endswith(".tif")])

# Preview first 3 images
for f in files[:20]:
  print(f"Image : {f}")
  preview_satellite_image_grid(os.path.join(folder, f))
  print("\n")


In [ ]:
input_data = sorted(os.listdir("/content/SatelliteDataset/images"))
labels_data = sorted(os.listdir("/content/SatelliteDataset/labels"))

filtered_labels = []

for i in os.listdir("/content/SatelliteDataset/labels"):
  if not("_" in i):
    filtered_labels.append(i)


filtered_labels = sorted(filtered_labels)

print(f"TIF Count : {len(input_data)} ---- Masks Count : {len(filtered_labels)}")

for i in range(10):

  print(f"Trainging : {input_data[i]} ---- Label : {filtered_labels[i]}")


In [ ]:
num_samples = 20


total_available = min(len(filtered_labels), len(input_data))
indices = random.sample(range(total_available), min(num_samples, total_available))


for i in indices:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    input_tif = input_data[i]
    input_png = filtered_labels[i]

    path_input = os.path.join("/content/SatelliteDataset/images" , input_tif)
    with rasterio.open(path_input) as src:
        image = src.read()

    # Normalize only RGB bands
    rgb = image[[3,2,1]].astype(np.float32)
    for c in range(3):
        band = rgb[c]
        rgb[c] = (band - band.min()) / (band.max() - band.min())
    rgb = np.transpose(rgb, (1, 2, 0))

    axes[0].imshow(rgb)
    axes[0].set_title(f"RGB Preview {input_tif}")

    path_png = os.path.join("/content/SatelliteDataset/labels" , input_png)

    # To view binary masks correctly, use gray colormap
    img = cv2.imread(path_png)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img * 255

    axes[1].imshow(img)
    axes[1].set_title(f"RGB Preview {input_png}")

    plt.tight_layout()
    plt.show()

In [ ]:
IMAGE_DIR = "/content/SatelliteDataset/images"
LABEL_DIR = "/content/SatelliteDataset/labels"

SELECTED_CHANNELS = [1,2,3,4,5,6,11]  # Blue, Green, Red, NIR, SWIR1, SWIR2, WaterProb


def load_dataset(input_data, filtered_labels):

    images = []
    masks = []

    for i in range(len(input_data)):

        tif_path = os.path.join(IMAGE_DIR, input_data[i])
        mask_path = os.path.join(LABEL_DIR, filtered_labels[i])

        # ---- Load TIF ----
        with rasterio.open(tif_path) as src:
            img = src.read()  # (12, H, W)

        img = img[SELECTED_CHANNELS, :, :].astype(np.float32)  # (7, H, W)

        # ---- Normalize spectral bands ----
        for c in range(6):  # first 6 = spectral
            band = img[c]
            min_val = band.min()
            max_val = band.max()
            if max_val > min_val:
                img[c] = (band - min_val) / (max_val - min_val)
            else:
                img[c] = 0

        # ---- Normalize water probability ----
        img[6] = img[6] / 100.0  # scale 0-100 → 0-1

        img = np.transpose(img, (1,2,0))  # (H, W, 7)

        # ---- Load mask ----
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)

        mask = np.expand_dims(mask, axis=-1)

        images.append(img)
        masks.append(mask)

    images = np.array(images, dtype=np.float32)
    masks = np.array(masks, dtype=np.float32)

    return images, masks


In [ ]:
X, y = load_dataset(input_data, filtered_labels)

print("Images shape:", X.shape)
print("Masks shape:", y.shape)


In [ ]:

def conv_block(x, filters):
    x = Conv2D(filters, 3, activation='relu', padding='same')(x)
    x = Conv2D(filters, 3, activation='relu', padding='same')(x)
    return x


def build_unet(input_shape=(128, 128, 7)):

    inputs = Input(input_shape)

    # -------- Encoder --------
    c1 = conv_block(inputs, 32)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, 64)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = conv_block(p2, 128)
    p3 = MaxPooling2D((2, 2))(c3)

    # -------- Bottleneck --------
    c4 = conv_block(p3, 256)

    # -------- Decoder --------
    u5 = Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(c4)
    u5 = concatenate([u5, c3])
    c5 = conv_block(u5, 128)

    u6 = Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(c5)
    u6 = concatenate([u6, c2])
    c6 = conv_block(u6, 64)

    u7 = Conv2DTranspose(32, 2, strides=(2, 2), padding='same')(c6)
    u7 = concatenate([u7, c1])
    c7 = conv_block(u7, 32)

    # -------- Output --------
    outputs = Conv2D(1, 1, activation='sigmoid')(c7)

    model = Model(inputs, outputs)

    return model


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = build_unet((X.shape[1], X.shape[2], 7))


model.summary()

tf.keras.utils.plot_model(
    model,
    show_shapes=True,
    show_layer_names=True,
    rankdir='LR',
)


In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (
        K.sum(y_true_f) + K.sum(y_pred_f) + smooth
    )


def iou_metric(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    total = K.sum(y_true_f) + K.sum(y_pred_f)
    union = total - intersection
    return (intersection + smooth) / (union + smooth)


def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)


def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)


Lighter Architecture and no NDWI

In [ ]:
model = build_unet((X.shape[1], X.shape[2], 7))

model.compile(
    optimizer='adam',
    loss=bce_dice_loss,
    metrics=[dice_coefficient, iou_metric]
)
history2 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=8
)


In [ ]:
def conv_block2(x, filters):
    x = tf.keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    x = tf.keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    return x

def build_unet2(input_shape=(128, 128, 7)):

    inputs = Input(input_shape)

    # -------- Encoder --------
    c1 = conv_block2(inputs, 32)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = conv_block2(p1, 64)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = conv_block2(p2, 128)
    p3 = MaxPooling2D((2, 2))(c3)

    # -------- Bottleneck --------
    c4 = conv_block2(p3, 256)
    c4 = Dropout(0.3)(c4)

    # -------- Decoder --------
    u5 = Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(c4)
    u5 = concatenate([u5, c3])
    c5 = conv_block2(u5, 128)

    u6 = Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(c5)
    u6 = concatenate([u6, c2])
    c6 = conv_block2(u6, 64)

    u7 = Conv2DTranspose(32, 2, strides=(2, 2), padding='same')(c6)
    u7 = concatenate([u7, c1])
    c7 = conv_block2(u7, 32)

    outputs = Conv2D(1, 1, activation='sigmoid')(c7)

    model = Model(inputs, outputs)

    return model


In [ ]:
model2 = build_unet2((X.shape[1], X.shape[2], 7))

model2.compile(
    optimizer='adam',
    loss=bce_dice_loss,
    metrics=[dice_coefficient, iou_metric]
)

model2.summary()


Complexer Architecture and no NDWI

In [ ]:

history3 = model2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=8
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def visualize_predictions(model, X, y, num_samples=3):

    preds = model.predict(X[:num_samples])
    preds = (preds > 0.5).astype(np.float32)

    for i in range(num_samples):

        # Reconstruct RGB from input
        rgb = X[i][:,:,:3]  # Blue, Green, Red
        rgb = rgb[..., ::-1]  # convert BGR → RGB for display

        plt.figure(figsize=(15,5))

        # Input RGB
        plt.subplot(1,3,1)
        plt.imshow(rgb)
        plt.title("Input RGB")
        plt.axis("off")

        # Ground Truth
        plt.subplot(1,3,2)
        plt.imshow(y[i].squeeze(), cmap='Blues')
        plt.title("Ground Truth")
        plt.axis("off")

        # Prediction
        plt.subplot(1,3,3)
        plt.imshow(preds[i].squeeze(), cmap='Blues')
        plt.title("Prediction")
        plt.axis("off")

        plt.show()
visualize_predictions(model, X_val, y_val, num_samples=5)


In [ ]:
visualize_predictions(model2, X_val, y_val, num_samples=5)
